# Lab 4 — 对 EEG 时间序列分类以识别癫痫

**目标：** 使用 CNN 对 EEG 小波图像进行癫痫/非癫痫二分类  
**关键技能：** 卷积神经网络用于复杂时间序列

## 工作流程
1. 从 PhysioNet 下载 CHB-MIT chb08 数据
2. 提取 EEG 片段 → 两阶段生成小波尺度图图像（全局归一化）
3. 构建并训练 CNN 分类器（轻量级架构 + 类别权重）
4. 评估并保存模型

In [1]:
# 1. Setup
from pathlib import Path
import re, time, urllib.request
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mne
from scipy.signal import butter, sosfiltfilt
import pywt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

BASE = Path.cwd()
DATA_DIR = BASE / 'data' / 'chb08'
DATA_DIR.mkdir(parents=True, exist_ok=True)
SZ_DIR = BASE / 'images' / 'seizure'
NS_DIR = BASE / 'images' / 'nonseizure'
SZ_DIR.mkdir(parents=True, exist_ok=True)
NS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = BASE / 'model'
MODEL_DIR.mkdir(exist_ok=True)
print('Setup OK')

Setup OK


d:\python\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. 数据下载

In [ ]:
# Download CHB-MIT chb08 from PhysioNet
BASE_URL = 'https://physionet.org/files/chbmit/1.0.0/chb08'
EDF_FILES = [f'chb08_{i:02d}.edf' for i in
             [2,3,4,5,10,11,13,21]]
SUMMARY_FILE = 'chb08-summary.txt'

def download(url, dest):
    if dest.exists():
        return
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as resp:
                dest.write_bytes(resp.read())
            print(f'  [OK] {dest.name}')
            return
        except Exception as e:
            if attempt == 2: print(f'  [FAIL] {dest.name}: {e}')
            else: time.sleep(5)

print('Downloading summary...')
download(f'{BASE_URL}/{SUMMARY_FILE}', DATA_DIR / SUMMARY_FILE)
for f in EDF_FILES:
    download(f'{BASE_URL}/{f}', DATA_DIR / f)
print('Done.')

## 2. 解析癫痫发作标注

In [ ]:
def parse_summary(path):
    text = path.read_text()
    seizures = {}
    current_file = None
    for line in text.split(chr(10)):
        m = re.match(r'File Name: (chb08_\d+\.edf)', line)
        if m:
            current_file = m.group(1)
            seizures[current_file] = []
        m = re.match(r'Seizure \d+ Start Time: (\d+) seconds', line)
        if m and current_file:
            start = int(m.group(1))
            seizures[current_file].append(start)
        m = re.match(r'Seizure \d+ End Time: (\d+) seconds', line)
        if m and current_file and seizures[current_file]:
            end = int(m.group(1))
            seizures[current_file][-1] = (seizures[current_file][-1], end)
    return {k: v for k, v in seizures.items() if v}

summary_path = DATA_DIR / 'chb08-summary.txt'
sz_map = parse_summary(summary_path)
print('Seizure annotations:')
for fname, intervals in sz_map.items():
    print(f'  {fname}: {intervals}')

## 3. EEG 处理 & 小波图像生成函数（两阶段全局归一化）

**关键改进：** 第一阶段计算所有片段的 wavelet power，第二阶段使用全局统一的 vmin/vmax 渲染图像，
确保癫痫与非癫痫图像之间的功率差异得以保留。

In [ ]:
# Constants
FS, CUTOFF = 256.0, 60.0
WIN_SEC, STEP_SEC, IMG_SIZE = 10, 5, 224
NON_SEIZURE_GAP = 300  # seconds away from any seizure

def load_edf(path):
    raw = mne.io.read_raw_edf(str(path), preload=True, verbose=False)
    pk = mne.pick_types(raw.info, eeg=True, exclude='bads')
    raw.pick(pk)
    return raw.get_data() * 1e6, raw.times

def avg_chan(data): return np.mean(data, axis=0)

def lowpass(x):
    ny = 0.5 * FS
    sos = butter(6, min(CUTOFF, ny*0.99)/ny, btype='low', output='sos')
    return sosfiltfilt(sos, x)

def wavelet_power(segment):
    freqs = np.linspace(1, 60, IMG_SIZE)
    cf = pywt.central_frequency('morl')
    scales = cf * FS / freqs
    co, _ = pywt.cwt(segment, scales, 'morl', sampling_period=1/FS)
    return 10 * np.log10(np.abs(co)**2 + 1e-12)

def save_wavelet_image(pw, fname, vmin, vmax):
    dpi = 100
    fig = plt.figure(figsize=(IMG_SIZE/dpi, IMG_SIZE/dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(pw, aspect='auto', origin='lower', cmap='jet',
              vmin=vmin, vmax=vmax, interpolation='bilinear')
    ax.axis('off')
    fig.savefig(fname, dpi=dpi, pad_inches=0)
    plt.close(fig)

def extract_seizure(data, times, intervals):
    segs, ws = [], int(WIN_SEC*FS)
    for lo, hi in intervals:
        for t0 in np.arange(lo, hi - WIN_SEC + 1, STEP_SEC):
            seg = data[:, int(t0*FS):int(t0*FS)+ws]
            if seg.shape[1] == ws: segs.append(seg)
    return segs

def extract_nonseizure(data, times, seizure_intervals, n_needed):
    """Random 10s windows far from any seizure region."""
    segs, ws = [], int(WIN_SEC*FS)
    total_sec = int(times[-1])

    blocked = []
    for s, e in seizure_intervals:
        blocked.append((max(0, s - NON_SEIZURE_GAP),
                        min(total_sec, e + NON_SEIZURE_GAP)))

    def _clean(t):
        for lo, hi in blocked:
            if lo <= t <= hi:
                return False
        return True

    rng = np.random.RandomState(42)
    attempts = 0
    while len(segs) < n_needed and attempts < 5000:
        t0 = rng.randint(0, max(0, total_sec - WIN_SEC))
        if _clean(t0):
            idx = int(t0 * FS)
            seg = data[:, idx:idx + ws]
            if seg.shape[1] == ws:
                segs.append(seg)
        attempts += 1
    return segs

print('Functions defined.')

## 4. 生成训练图像（两阶段：先计算功率，再全局归一化渲染）

In [ ]:
# Clean old images to avoid pollution from previous runs
import shutil
for d in [SZ_DIR, NS_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
print('Old images cleaned.')

# Phase 1: Compute wavelet power for all segments
# Use all available EDF files on disk (same as generate_images.py)
available = sorted(DATA_DIR.glob('chb08_*.edf'))
print(f'Available EDF files: {len(available)}')
sz_files = [f for f in available if f.name in sz_map]
ns_files = [f for f in available if f.name not in sz_map]
print(f'Seizure files: {len(sz_files)}, Non-seizure files: {len(ns_files)}')

records = []  # (power_matrix, label, output_path)

# ---- Seizure ----
print('\n--- Phase 1: Seizure ---')
for edf in sz_files:
    data, times = load_edf(edf)
    segs = extract_seizure(data, times, sz_map[edf.name])
    for i, seg in enumerate(segs):
        av_f = lowpass(avg_chan(seg))
        pw = wavelet_power(av_f)
        records.append((pw, 'seizure', SZ_DIR / f'{edf.stem}_{i:03d}.png'))
    print(f'  {edf.name}: {len(segs)} images')

# ---- Non-seizure ----
print('\n--- Phase 1: Non-seizure ---')
n_each = max(4, 150 // max(1, len(ns_files)))
for edf in ns_files:
    data, times = load_edf(edf)
    segs = extract_nonseizure(data, times, sz_map.get(edf.name, []), n_each)
    for i, seg in enumerate(segs):
        av_f = lowpass(avg_chan(seg))
        pw = wavelet_power(av_f)
        records.append((pw, 'nonseizure', NS_DIR / f'{edf.stem}_{i:03d}.png'))
    print(f'  {edf.name}: {len(segs)} images')

# Phase 2: Global vmin/vmax from seizure images
sz_powers = [pw for pw, lbl, _ in records if lbl == 'seizure']
if sz_powers:
    all_sz = np.concatenate([p.ravel() for p in sz_powers])
    vmin, vmax = np.percentile(all_sz, 5), np.percentile(all_sz, 99)
else:
    all_pw = np.concatenate([p.ravel() for p, _, _ in records])
    vmin, vmax = np.percentile(all_pw, 5), np.percentile(all_pw, 99)
print(f'\nGlobal vmin={vmin:.2f}, vmax={vmax:.2f}')

# Phase 2: Save all images with global scale
sz_total = ns_total = 0
for pw, lbl, out_path in records:
    save_wavelet_image(pw, out_path, vmin, vmax)
    if lbl == 'seizure': sz_total += 1
    else: ns_total += 1
print(f'\nDataset: {sz_total} seizure + {ns_total} non-seizure = {sz_total+ns_total} total')

## 5. 加载数据集（128×128 输入）

In [ ]:
INPUT_SIZE = 128  # smaller input → fewer parameters → less overfitting

def load_dataset():
    def _load(d, lbl):
        imgs, labs = [], []
        for f in sorted(d.glob('*.png')):
            img = tf.keras.utils.load_img(str(f), target_size=(INPUT_SIZE, INPUT_SIZE))
            imgs.append(tf.keras.utils.img_to_array(img))
            labs.append(lbl)
        return imgs, labs
    sz_i, sz_l = _load(SZ_DIR, 1)
    ns_i, ns_l = _load(NS_DIR, 0)
    X = np.array(sz_i + ns_i, dtype=np.float32) / 255.0
    y = np.array(sz_l + ns_l, dtype=np.float32)
    print(f'X: {X.shape}, y: {y.shape}')
    print(f'Classes: {np.bincount(y.astype(int))}')
    return X, y

X, y = load_dataset()

## 6. 训练/测试划分 & 数据增强

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# Class weights to handle potential imbalance
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = {int(k): float(v) for k, v in zip(classes, cw)}
print(f'Class weights: {class_weight}')

augmenter = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.15, 0.15),
])

BS = 16
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(512).batch(BS).map(
    lambda x, y: (augmenter(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(BS).prefetch(tf.data.AUTOTUNE)
print('Dataset ready.')

## 7. 构建 CNN 模型（轻量级架构 + BatchNorm）

采用 3 层卷积 + BatchNormalization 的轻量级设计，
适配小样本数据集，减少过拟合风险。

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

reg = keras.regularizers.l2(2e-4)

model = keras.Sequential([
    layers.Input((INPUT_SIZE, INPUT_SIZE, 3)),

    layers.Conv2D(24, 3, padding='same', activation='relu',
                  kernel_regularizer=reg),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),

    layers.Conv2D(48, 3, padding='same', activation='relu',
                  kernel_regularizer=reg),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),

    layers.Conv2D(96, 3, padding='same', activation='relu',
                  kernel_regularizer=reg),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.6),
    layers.Dense(64, activation='relu', kernel_regularizer=reg),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid'),
])

model.summary()

## 8. 训练模型

In [ ]:
EPOCHS = 100
steps_per_epoch = max(1, len(X_train) // BS)
total_steps = steps_per_epoch * EPOCHS
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=2e-4,
    decay_steps=total_steps,
    alpha=0.025,
)
print(f'Steps/epoch: {steps_per_epoch}, Total decay steps: {total_steps}')
print(f'LR: {2e-4:.1e} -> {2e-4 * 0.025:.1e} over {EPOCHS} epochs')

model.compile(optimizer=keras.optimizers.Adam(lr_schedule),
              loss='binary_crossentropy', metrics=['accuracy'])

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / 'best.keras'), save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True),
]

history = model.fit(
    train_ds, validation_data=test_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1,
    class_weight=class_weight)

## 9. 评估 & 可视化

In [ ]:
# Evaluate
y_pred = (model.predict(test_ds) > 0.5).astype(int).flatten()
acc = np.mean(y_pred == y_test)
f1 = f1_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')
print(f'Test F1 Score: {f1:.4f}')
print(classification_report(y_test, y_pred, target_names=['Non-seizure', 'Seizure']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(['Non-seizure','Seizure'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Non-seizure','Seizure'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (acc={acc:.3f}, f1={f1:.3f})')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title(f'Loss Curves (final acc={acc:.3f}, f1={f1:.3f})')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history['accuracy'], label='Train')
ax2.plot(history.history['val_accuracy'], label='Test')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'Accuracy Curves (final acc={acc:.3f}, f1={f1:.3f})')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves.png', dpi=150)
plt.show()

# Save final model
model.save(str(MODEL_DIR / 'final_model.keras'))
print('Model saved.')